<a href="https://colab.research.google.com/github/bhushan191004/LoanDesk-RAG-Loan-Assistant/blob/main/loanassistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain-groq langchain-community langchain-text-splitters sentence-transformers chromadb pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 113.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/

In [9]:
import gradio as gr
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings


import os

groq_api_key =("insert your groq key here")

if not groq_api_key:
    raise ValueError("Please set the GROQ_API_KEY environment variable.")

llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    groq_api_key=groq_api_key,
    temperature=0
)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = None


theme = gr.themes.Ocean(
    primary_hue="blue",
    secondary_hue="cyan",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"],
).set(
    button_primary_background_fill="*primary_600",
    button_primary_background_fill_hover="*primary_500",
    block_title_text_weight="600",
    container_radius="lg",
    button_large_radius="md"
)


custom_css = """
.gradio-container { background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%); }
.sidebar { background: rgba(30, 41, 59, 0.7) !important; border-right: 1px solid rgba(255,255,255,0.1); backdrop-filter: blur(10px); }
#title-header { text-align: center; margin-bottom: 20px; color: white; }
#status-box { background: rgba(15, 23, 42, 0.5); border: 1px solid #3b82f6; border-radius: 8px; padding: 10px; color: #60a5fa; }
.main-chat-box { border-radius: 15px !important; box-shadow: 0 4px 30px rgba(0, 0, 0, 0.1); }
"""


def process_and_chat(file, message):
    global vectorstore
    if vectorstore is None and file is not None:
        try:
            loader = PyPDFLoader(file.name)
            docs = loader.load()
            text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=50)
            splits = text_splitter.split_documents(docs)
            vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
        except Exception as e: return f"Error: {str(e)}"

    if vectorstore is None: return "Please upload a document first."

    docs = vectorstore.similarity_search(message, k=2)
    context = "\n".join([d.page_content for d in docs])
    prompt = f"Context: {context}\n\nQuestion: {message}\n\nAnswer like a senior financial analyst:"

    try:
        response = llm.invoke(prompt)
        return response.content
    except Exception as e: return f"Groq Error: {str(e)}"


with gr.Blocks(theme=theme, css=custom_css) as demo:
    with gr.Row(elem_id="title-header"):
        gr.Markdown("# 🏦LOANDESK :LOAN PRODUCT AI ASSISTANT:USING LLAMA 3.3")
        gr.Markdown("#### *Institutional-grade PDF analysis powered by Llama 3.3 70B*")

    with gr.Row():

        with gr.Column(scale=1, variant="panel", elem_classes="sidebar"):
            gr.Markdown("### 📥 Document Center")
            file_input = gr.File(label="Upload Agreement (PDF)", file_types=[".pdf"])
            gr.Markdown("---")
            status = gr.Markdown("**System Status:** Ready", elem_id="status-box")
            gr.Markdown("💡 *Tip: Ask about late fees, cooling-off periods, or interest compounding.*")


        with gr.Column(scale=3):
            with gr.Group(elem_classes="main-chat-box"):
                chat_input = gr.Textbox(
                    label="Analysis Query",
                    placeholder="Type your question about the loan here...",
                    lines=2
                )
                output = gr.Textbox(
                    label="Financial Analyst Report",
                    placeholder="AI insights will appear here...",
                    lines=12,
                    interactive=False
                )
                with gr.Row():
                    clear_btn = gr.Button("Clear", variant="secondary")
                    chat_btn = gr.Button("Run Analysis", variant="primary")


    chat_btn.click(fn=process_and_chat, inputs=[file_input, chat_input], outputs=output)
    clear_btn.click(lambda: (None, ""), None, [chat_input, output])

demo.launch(share=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/tmp/ipykernel_11337/3363556235.py:71: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, css=custom_css) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/06/30 18:50:21 [W] [service.go:132] login to server failed: session shutdown


<IPython.core.display.Javascript object>